In [19]:
import sys

print("Python version:")
print(sys.version)

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [20]:
from pathlib import Path

folders = [
    "project/data/raw",
    "project/data/processed",
    "project/src/preprocessing",
    "project/src/models",
    "project/src/xai",
    "project/src/evaluation",
    "project/src/statistics",
    "project/notebooks",
    "project/results/xai_outputs",
    "project/figures",
    "project/paper"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folder structure created successfully.")

Project folder structure created successfully.


In [21]:
!pip install -q torch shap lime dice-ml xgboost scikit-learn pandas numpy scipy scikit-posthocs pingouin matplotlib seaborn kaggle


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import torch
import shap
import lime
import dice_ml
import xgboost
import sklearn
import scipy
import scikit_posthocs
import pingouin
import matplotlib
import seaborn
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("All required libraries imported successfully.")
print("PyTorch version:", torch.__version__)

All required libraries imported successfully.
PyTorch version: 2.13.0+cpu


In [23]:
from pathlib import Path

dataset_path = Path(
    "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"
)

print("Exists:", dataset_path.exists())
print("Path:", dataset_path.resolve())

Exists: True
Path: C:\Users\DELL\Desktop\DL research paper\project\data\raw\diabetes_binary_health_indicators_BRFSS2015.csv


In [24]:
from pathlib import Path
import pandas as pd

dataset_path = Path(
    "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"
)

print("Dataset exists:", dataset_path.exists())
print("Dataset path:", dataset_path.resolve())

df_check = pd.read_csv(dataset_path)

print("\nDataset shape:", df_check.shape)
print("Columns:", len(df_check.columns))
print("Missing values:", df_check.isna().sum().sum())

Dataset exists: True
Dataset path: C:\Users\DELL\Desktop\DL research paper\project\data\raw\diabetes_binary_health_indicators_BRFSS2015.csv

Dataset shape: (253680, 22)
Columns: 22
Missing values: 0


In [25]:
from pathlib import Path

# Define the existing dataset location
dataset_path = Path(
    "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"
)

# Verify that the dataset exists
if dataset_path.exists():
    print("Dataset found successfully!")
    print("Location:", dataset_path.resolve())
else:
    print("ERROR: Dataset not found!")
    print("Expected location:", dataset_path.resolve())

Dataset found successfully!
Location: C:\Users\DELL\Desktop\DL research paper\project\data\raw\diabetes_binary_health_indicators_BRFSS2015.csv


In [26]:
import pandas as pd

file_path = "project/data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (253680, 22)

Columns:
['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


In [27]:
# Check the target distribution

target_counts = df["Diabetes_binary"].value_counts().sort_index()
target_percent = df["Diabetes_binary"].value_counts(normalize=True).sort_index() * 100

print("Diabetes_binary distribution:")
print(target_counts)

print("\nPercentage distribution:")
print(target_percent)

print("\nClass labels:")
print("0 = No diabetes")
print("1 = Diabetes")

Diabetes_binary distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Percentage distribution:
Diabetes_binary
0.0    86.066698
1.0    13.933302
Name: proportion, dtype: float64

Class labels:
0 = No diabetes
1 = Diabetes


In [28]:
# Check for missing values in every column

missing_values = df.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values in dataset:", missing_values.sum())

if missing_values.sum() == 0:
    print("✓ No missing values found.")
else:
    print("⚠ Missing values found.")

Missing values per column:
Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Total missing values in dataset: 0
✓ No missing values found.


In [29]:
# Check data types and number of unique values

print("DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n\nUNIQUE VALUES PER COLUMN")
print("=" * 50)
print(df.nunique().sort_values())

DATA TYPES
Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


UNIQUE VALUES PER COLUMN
Diabetes_binary          2
HighBP                   2
HighChol                 2
CholCheck                2
Smoker                   2
Stroke                   2
HeartDiseaseorAttack     2
PhysActivity             2
AnyHealthcare            2
F

In [30]:
# Inspect the actual unique values of important features

features_to_check = [
    "HighBP",
    "Sex",
    "GenHlth",
    "Age",
    "Education",
    "Income",
    "BMI",
    "MentHlth",
    "PhysHlth"
]

for feature in features_to_check:
    print(f"\n{feature}")
    print("-" * 40)
    print(sorted(df[feature].unique()))


HighBP
----------------------------------------
[np.float64(0.0), np.float64(1.0)]

Sex
----------------------------------------
[np.float64(0.0), np.float64(1.0)]

GenHlth
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

Age
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0)]

Education
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]

Income
----------------------------------------
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0)]

BMI
----------------------------------------
[np.float64(12.0), np.float64(13.0), np.

In [31]:
# Descriptive statistics for all columns

pd.set_option("display.max_columns", None)

print("DESCRIPTIVE STATISTICS")
print("=" * 80)

print(df.describe().T)

DESCRIPTIVE STATISTICS
                         count       mean       std   min   25%   50%   75%  \
Diabetes_binary       253680.0   0.139333  0.346294   0.0   0.0   0.0   0.0   
HighBP                253680.0   0.429001  0.494934   0.0   0.0   0.0   1.0   
HighChol              253680.0   0.424121  0.494210   0.0   0.0   0.0   1.0   
CholCheck             253680.0   0.962670  0.189571   0.0   1.0   1.0   1.0   
BMI                   253680.0  28.382364  6.608694  12.0  24.0  27.0  31.0   
Smoker                253680.0   0.443169  0.496761   0.0   0.0   0.0   1.0   
Stroke                253680.0   0.040571  0.197294   0.0   0.0   0.0   0.0   
HeartDiseaseorAttack  253680.0   0.094186  0.292087   0.0   0.0   0.0   0.0   
PhysActivity          253680.0   0.756544  0.429169   0.0   1.0   1.0   1.0   
Fruits                253680.0   0.634256  0.481639   0.0   0.0   1.0   1.0   
Veggies               253680.0   0.811420  0.391175   0.0   1.0   1.0   1.0   
HvyAlcoholConsump     253680.

In [32]:
# Correlation of each feature with the diabetes target

correlations = df.corr(numeric_only=True)["Diabetes_binary"].drop("Diabetes_binary")

correlations = correlations.sort_values(ascending=False)

print("Feature correlation with Diabetes_binary:")
print("=" * 60)
print(correlations)

Feature correlation with Diabetes_binary:
GenHlth                 0.293569
HighBP                  0.263129
DiffWalk                0.218344
BMI                     0.216843
HighChol                0.200276
Age                     0.177442
HeartDiseaseorAttack    0.177282
PhysHlth                0.171337
Stroke                  0.105816
MentHlth                0.069315
CholCheck               0.064761
Smoker                  0.060789
NoDocbcCost             0.031433
Sex                     0.031430
AnyHealthcare           0.016255
Fruits                 -0.040779
Veggies                -0.056584
HvyAlcoholConsump      -0.057056
PhysActivity           -0.118133
Education              -0.124456
Income                 -0.163919
Name: Diabetes_binary, dtype: float64


In [33]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: divide the 30% temporary data equally
# This gives 15% validation and 15% test overall
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Dataset split completed.\n")

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Dataset split completed.

Training set: (177576, 21) (177576,)
Validation set: (38052, 21) (38052,)
Test set: (38052, 21) (38052,)


In [34]:
stratify=y

In [35]:
# Verify class distribution in each split

def show_distribution(name, target):
    counts = target.value_counts().sort_index()
    percentages = target.value_counts(normalize=True).sort_index() * 100

    print(f"\n{name}")
    print("-" * 50)

    for class_value in counts.index:
        print(
            f"Class {int(class_value)}: "
            f"{counts[class_value]:,} samples "
            f"({percentages[class_value]:.2f}%)"
        )

show_distribution("TRAINING SET", y_train)
show_distribution("VALIDATION SET", y_val)
show_distribution("TEST SET", y_test)


TRAINING SET
--------------------------------------------------
Class 0: 152,834 samples (86.07%)
Class 1: 24,742 samples (13.93%)

VALIDATION SET
--------------------------------------------------
Class 0: 32,750 samples (86.07%)
Class 1: 5,302 samples (13.93%)

TEST SET
--------------------------------------------------
Class 0: 32,750 samples (86.07%)
Class 1: 5,302 samples (13.93%)


In [36]:
from sklearn.preprocessing import StandardScaler

# Features specified by the project for standardization
scaled_features = [
    "BMI",
    "MentHlth",
    "PhysHlth",
    "Age"
]

# Create the scaler
scaler = StandardScaler()

# Fit ONLY on the training data
scaler.fit(X_train[scaled_features])

print("Scaler fitted successfully.")
print("\nFeatures being standardized:")
print(scaled_features)

print("\nTraining means learned by scaler:")
print(scaler.mean_)

print("\nTraining standard deviations learned by scaler:")
print(scaler.scale_)

Scaler fitted successfully.

Features being standardized:
['BMI', 'MentHlth', 'PhysHlth', 'Age']

Training means learned by scaler:
[28.37629522  3.19317926  4.24358021  8.03167095]

Training standard deviations learned by scaler:
[6.60785624 7.42707453 8.71942756 3.05035369]


In [37]:
scaler.fit(X_train[scaled_features])

StandardScaler()

In [38]:
# Create copies so the original split data remains unchanged
X_train_processed = X_train.copy()
X_val_processed = X_val.copy()
X_test_processed = X_test.copy()

# Apply the training-fitted scaler to all three datasets
X_train_processed[scaled_features] = scaler.transform(
    X_train[scaled_features]
)

X_val_processed[scaled_features] = scaler.transform(
    X_val[scaled_features]
)

X_test_processed[scaled_features] = scaler.transform(
    X_test[scaled_features]
)

print("Preprocessing transformation completed successfully.")
print("\nTraining shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Test shape:", X_test_processed.shape)

Preprocessing transformation completed successfully.

Training shape: (177576, 21)
Validation shape: (38052, 21)
Test shape: (38052, 21)


In [39]:
# Verify that the training data was standardized correctly

print("TRAINING SET AFTER STANDARDIZATION")
print("=" * 60)

for feature in scaled_features:
    print(
        f"{feature:10s} | "
        f"mean = {X_train_processed[feature].mean():.6f} | "
        f"std = {X_train_processed[feature].std():.6f}"
    )

TRAINING SET AFTER STANDARDIZATION
BMI        | mean = 0.000000 | std = 1.000003
MentHlth   | mean = 0.000000 | std = 1.000003
PhysHlth   | mean = -0.000000 | std = 1.000003
Age        | mean = 0.000000 | std = 1.000003


In [40]:
import joblib
from pathlib import Path

# Create model/preprocessing directory
Path("project/src/models").mkdir(parents=True, exist_ok=True)

# Save the fitted scaler
scaler_path = "project/src/models/scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Scaler saved successfully!")
print("Saved to:", scaler_path)

Scaler saved successfully!
Saved to: project/src/models/scaler.pkl


In [41]:
# Verify that the saved scaler can be loaded

loaded_scaler = joblib.load(scaler_path)

print("Saved scaler loaded successfully!")
print("Scaled features:", loaded_scaler.feature_names_in_)

Saved scaler loaded successfully!
Scaled features: ['BMI' 'MentHlth' 'PhysHlth' 'Age']


In [42]:
import numpy as np

# Convert target labels to float32
y_train_processed = y_train.to_numpy(dtype=np.float32)
y_val_processed = y_val.to_numpy(dtype=np.float32)
y_test_processed = y_test.to_numpy(dtype=np.float32)

print("Labels prepared successfully.")

print("\nTraining labels:")
print("Shape:", y_train_processed.shape)
print("Data type:", y_train_processed.dtype)
print("Unique values:", np.unique(y_train_processed))

print("\nValidation labels:")
print("Shape:", y_val_processed.shape)
print("Data type:", y_val_processed.dtype)
print("Unique values:", np.unique(y_val_processed))

print("\nTest labels:")
print("Shape:", y_test_processed.shape)
print("Data type:", y_test_processed.dtype)
print("Unique values:", np.unique(y_test_processed))

Labels prepared successfully.

Training labels:
Shape: (177576,)
Data type: float32
Unique values: [0. 1.]

Validation labels:
Shape: (38052,)
Data type: float32
Unique values: [0. 1.]

Test labels:
Shape: (38052,)
Data type: float32
Unique values: [0. 1.]


In [43]:
# Convert processed feature DataFrames to NumPy arrays

X_train_np = X_train_processed.to_numpy(dtype=np.float32)
X_val_np = X_val_processed.to_numpy(dtype=np.float32)
X_test_np = X_test_processed.to_numpy(dtype=np.float32)

print("Feature arrays created successfully.")

print("\nTraining:")
print("Shape:", X_train_np.shape)
print("Data type:", X_train_np.dtype)

print("\nValidation:")
print("Shape:", X_val_np.shape)
print("Data type:", X_val_np.dtype)

print("\nTest:")
print("Shape:", X_test_np.shape)
print("Data type:", X_test_np.dtype)

Feature arrays created successfully.

Training:
Shape: (177576, 21)
Data type: float32

Validation:
Shape: (38052, 21)
Data type: float32

Test:
Shape: (38052, 21)
Data type: float32


In [44]:
# Final preprocessing sanity check

print("FINAL PREPROCESSING SANITY CHECK")
print("=" * 60)

# Check NaN values
print("\nNaN values:")
print("Train:", np.isnan(X_train_np).sum())
print("Validation:", np.isnan(X_val_np).sum())
print("Test:", np.isnan(X_test_np).sum())

# Check infinite values
print("\nInfinite values:")
print("Train:", np.isinf(X_train_np).sum())
print("Validation:", np.isinf(X_val_np).sum())
print("Test:", np.isinf(X_test_np).sum())

# Check overall numerical ranges
print("\nOverall feature ranges:")
print("Train min:", X_train_np.min())
print("Train max:", X_train_np.max())
print("Validation min:", X_val_np.min())
print("Validation max:", X_val_np.max())
print("Test min:", X_test_np.min())
print("Test max:", X_test_np.max())

FINAL PREPROCESSING SANITY CHECK

NaN values:
Train: 0
Validation: 0
Test: 0

Infinite values:
Train: 0
Validation: 0
Test: 0

Overall feature ranges:
Train min: -2.4783068
Train max: 10.536504
Validation min: -2.4783068
Validation max: 10.0824995
Test min: -2.4783068
Test max: 10.536504


In [45]:
import torch

# Convert feature arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)

# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train_processed, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_processed, dtype=torch.float32)

print("PyTorch tensors created successfully.")

print("\nTraining:")
print("X:", X_train_tensor.shape, X_train_tensor.dtype)
print("y:", y_train_tensor.shape, y_train_tensor.dtype)

print("\nValidation:")
print("X:", X_val_tensor.shape, X_val_tensor.dtype)
print("y:", y_val_tensor.shape, y_val_tensor.dtype)

print("\nTest:")
print("X:", X_test_tensor.shape, X_test_tensor.dtype)
print("y:", y_test_tensor.shape, y_test_tensor.dtype)

PyTorch tensors created successfully.

Training:
X: torch.Size([177576, 21]) torch.float32
y: torch.Size([177576]) torch.float32

Validation:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32

Test:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32


In [46]:
import torch.nn as nn

class DiabetesDNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(21, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Second hidden layer
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Third hidden layer
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output layer
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [47]:
# Create the DNN model
model = DiabetesDNN()

print(model)

DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [48]:
# Count trainable parameters

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

print("\nParameters by layer:")
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(f"{name:30s} {parameter.numel():,}")

Trainable parameters: 13185

Parameters by layer:
network.0.weight               2,688
network.0.bias                 128
network.3.weight               8,192
network.3.bias                 64
network.6.weight               2,048
network.6.bias                 32
network.9.weight               32
network.9.bias                 1


In [49]:
import numpy as np

# Count classes in the training set
class_counts = np.bincount(y_train_processed.astype(int))

# Calculate balanced class weights
num_samples = len(y_train_processed)
num_classes = len(class_counts)

class_weights = num_samples / (num_classes * class_counts)

print("Training class counts:")
print("Class 0:", class_counts[0])
print("Class 1:", class_counts[1])

print("\nCalculated class weights:")
print("Class 0:", class_weights[0])
print("Class 1:", class_weights[1])

Training class counts:
Class 0: 152834
Class 1: 24742

Calculated class weights:
Class 0: 0.5809440307784917
Class 1: 3.5885538760003235


In [50]:
import torch
import torch.nn as nn

# Convert the positive-class weight to a PyTorch tensor
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

# Weighted binary cross-entropy loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("Weighted loss function created successfully.")
print("Positive-class weight:", pos_weight.item())
print("Loss function:", criterion)

Weighted loss function created successfully.
Positive-class weight: 3.5885539054870605
Loss function: BCEWithLogitsLoss()


In [51]:
import torch.optim as optim

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", 1e-3)

Optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [52]:

from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Create DataLoaders
batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DataLoaders created successfully.")

print("\nBatch size:", batch_size)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully.

Batch size: 256
Training batches: 694
Validation batches: 149
Test batches: 149


In [53]:
# Inspect one batch from the training DataLoader

X_batch, y_batch = next(iter(train_loader))

print("ONE TRAINING BATCH")
print("=" * 60)

print("Feature batch shape:", X_batch.shape)
print("Feature data type:", X_batch.dtype)

print("\nLabel batch shape:", y_batch.shape)
print("Label data type:", y_batch.dtype)

print("\nFirst patient's features:")
print(X_batch[0])

print("\nFirst 10 labels:")
print(y_batch[:10])

ONE TRAINING BATCH
Feature batch shape: torch.Size([256, 21])
Feature data type: torch.float32

Label batch shape: torch.Size([256])
Label data type: torch.float32

First patient's features:
tensor([ 1.0000,  1.0000,  1.0000, -0.5110,  1.0000,  0.0000,  0.0000,  1.0000,
         1.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000, -0.4299, -0.4867,
         0.0000,  0.0000, -0.6660,  6.0000,  8.0000])

First 10 labels:
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])


In [54]:
# Put the model in training mode
model.train()

# Perform one forward pass
logits = model(X_batch)

print("FORWARD PASS")
print("=" * 60)

print("Input shape:", X_batch.shape)
print("Output shape:", logits.shape)

print("\nFirst 10 raw logits:")
print(logits[:10].squeeze())

# Convert logits to probabilities for inspection
probabilities = torch.sigmoid(logits)

print("\nFirst 10 probabilities:")
print(probabilities[:10].squeeze())

FORWARD PASS
Input shape: torch.Size([256, 21])
Output shape: torch.Size([256, 1])

First 10 raw logits:
tensor([-0.3025, -0.2532, -0.1269, -0.3719, -0.4149, -0.1179, -0.3922, -0.0715,
        -0.2819, -0.0582], grad_fn=<SqueezeBackward0>)

First 10 probabilities:
tensor([0.4250, 0.4370, 0.4683, 0.4081, 0.3977, 0.4706, 0.4032, 0.4821, 0.4300,
        0.4855], grad_fn=<SqueezeBackward0>)


In [55]:
# Calculate the initial loss for the current batch

# Remove the final dimension from logits
# [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate weighted binary cross-entropy loss
initial_loss = criterion(batch_logits, y_batch)

print("INITIAL LOSS")
print("=" * 60)
print("Loss:", initial_loss.item())

INITIAL LOSS
Loss: 1.0358978509902954


In [56]:
# Perform one complete training step

# Make sure the model is in training mode
model.train()

# Clear gradients from any previous step
optimizer.zero_grad()

# Forward pass
logits = model(X_batch)

# Remove the final dimension: [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate loss
loss = criterion(batch_logits, y_batch)

# Backpropagation
loss.backward()

# Update model parameters
optimizer.step()

print("ONE TRAINING STEP COMPLETED")
print("=" * 60)
print("Loss before parameter update:", loss.item())

ONE TRAINING STEP COMPLETED
Loss before parameter update: 1.0345157384872437


In [57]:
# Check whether the model parameters were updated

print("PARAMETER UPDATE CHECK")
print("=" * 60)

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(
            f"{name:30s} | "
            f"mean = {parameter.data.mean().item():.6f} | "
            f"grad mean = {parameter.grad.mean().item():.6f}"
        )

PARAMETER UPDATE CHECK
network.0.weight               | mean = -0.002806 | grad mean = 0.000032
network.0.bias                 | mean = -0.004851 | grad mean = 0.000022
network.3.weight               | mean = 0.000527 | grad mean = 0.000081
network.3.bias                 | mean = 0.011600 | grad mean = 0.000256
network.6.weight               | mean = 0.003540 | grad mean = 0.000058
network.6.bias                 | mean = 0.010312 | grad mean = 0.000484
network.9.weight               | mean = -0.027019 | grad mean = 0.002310
network.9.bias                 | mean = -0.062738 | grad mean = -0.006263


In [58]:
# Reset the model before real training

model = DiabetesDNN()

# Recreate the weighted loss
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Recreate the optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Model reset successfully.")
print("\nArchitecture:")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

print("\nOptimizer:")
print(optimizer.__class__.__name__)

print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Positive-class weight:", pos_weight.item())

Model reset successfully.

Architecture:
DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters:
13185

Optimizer:
Adam
Learning rate: 0.001
Positive-class weight: 3.5885539054870605


In [59]:
from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import torch

# Training configuration
max_epochs = 100
patience = 10

# Store training history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

# Early stopping variables
best_val_auc = -np.inf
epochs_without_improvement = 0
best_model_state = None

print("Starting DNN training...")
print("=" * 70)

for epoch in range(max_epochs):

    # ============================================================
    # TRAINING
    # ============================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch).squeeze(1)

        # Calculate weighted loss
        loss = criterion(logits, y_batch)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate loss
        batch_size_actual = X_batch.size(0)
        running_train_loss += loss.item() * batch_size_actual
        train_samples += batch_size_actual

    epoch_train_loss = running_train_loss / train_samples

    # ============================================================
    # VALIDATION
    # ============================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    val_probabilities = []
    val_true_labels = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            # Forward pass
            logits = model(X_batch).squeeze(1)

            # Validation loss
            loss = criterion(logits, y_batch)

            # Convert logits to probabilities
            probabilities = torch.sigmoid(logits)

            # Accumulate validation loss
            batch_size_actual = X_batch.size(0)
            running_val_loss += loss.item() * batch_size_actual
            val_samples += batch_size_actual

            # Store predictions and true labels
            val_probabilities.extend(
                probabilities.cpu().numpy()
            )

            val_true_labels.extend(
                y_batch.cpu().numpy()
            )

    epoch_val_loss = running_val_loss / val_samples

    # Calculate validation ROC-AUC
    epoch_val_auc = roc_auc_score(
        val_true_labels,
        val_probabilities
    )

    # Store history
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_auc"].append(epoch_val_auc)

    # ============================================================
    # EARLY STOPPING / CHECKPOINTING
    # ============================================================

    if epoch_val_auc > best_val_auc:

        # Validation AUC improved
        best_val_auc = epoch_val_auc
        epochs_without_improvement = 0

        # Save a copy of the best model parameters
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        checkpoint_status = " ← BEST"

    else:

        # No improvement
        epochs_without_improvement += 1

        checkpoint_status = (
            f" | patience: "
            f"{epochs_without_improvement}/{patience}"
        )

    # ============================================================
    # PRINT EPOCH RESULTS
    # ============================================================

    print(
        f"Epoch {epoch + 1:03d}/{max_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val ROC-AUC: {epoch_val_auc:.4f}"
        f"{checkpoint_status}"
    )

    # ============================================================
    # EARLY STOPPING
    # ============================================================

    if epochs_without_improvement >= patience:

        print("\nEarly stopping triggered.")
        print(
            f"No validation ROC-AUC improvement for "
            f"{patience} consecutive epochs."
        )

        break


# ================================================================
# RESTORE BEST MODEL
# ================================================================

if best_model_state is not None:

    model.load_state_dict(best_model_state)

    print("\nBest model restored successfully.")
    print(f"Best validation ROC-AUC: {best_val_auc:.4f}")

print("\nTraining completed.")

Starting DNN training...
Epoch 001/100 | Train Loss: 0.7015 | Val Loss: 0.6749 | Val ROC-AUC: 0.8221 ← BEST
Epoch 002/100 | Train Loss: 0.6756 | Val Loss: 0.6676 | Val ROC-AUC: 0.8243 ← BEST
Epoch 003/100 | Train Loss: 0.6735 | Val Loss: 0.6653 | Val ROC-AUC: 0.8254 ← BEST
Epoch 004/100 | Train Loss: 0.6719 | Val Loss: 0.6654 | Val ROC-AUC: 0.8254 ← BEST
Epoch 005/100 | Train Loss: 0.6687 | Val Loss: 0.6650 | Val ROC-AUC: 0.8263 ← BEST
Epoch 006/100 | Train Loss: 0.6682 | Val Loss: 0.6638 | Val ROC-AUC: 0.8264 ← BEST
Epoch 007/100 | Train Loss: 0.6680 | Val Loss: 0.6639 | Val ROC-AUC: 0.8261 | patience: 1/10
Epoch 008/100 | Train Loss: 0.6660 | Val Loss: 0.6639 | Val ROC-AUC: 0.8267 ← BEST
Epoch 009/100 | Train Loss: 0.6666 | Val Loss: 0.6631 | Val ROC-AUC: 0.8272 ← BEST
Epoch 010/100 | Train Loss: 0.6660 | Val Loss: 0.6623 | Val ROC-AUC: 0.8273 ← BEST
Epoch 011/100 | Train Loss: 0.6655 | Val Loss: 0.6656 | Val ROC-AUC: 0.8270 | patience: 1/10
Epoch 012/100 | Train Loss: 0.6652 | Val L

In [60]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Put the restored best model into evaluation mode
model.eval()

test_probabilities = []
test_true_labels = []

# Generate predictions without calculating gradients
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Forward pass
        logits = model(X_batch).squeeze(1)

        # Convert logits to probabilities
        probabilities = torch.sigmoid(logits)

        # Store predictions and true labels
        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

        test_true_labels.extend(
            y_batch.cpu().numpy()
        )

# Convert to NumPy arrays
test_probabilities = np.array(test_probabilities)
test_true_labels = np.array(test_true_labels)

print("TEST PREDICTIONS GENERATED")
print("=" * 60)

print("Number of predictions:", len(test_probabilities))
print("Number of true labels:", len(test_true_labels))

print("\nProbability range:")
print("Minimum:", test_probabilities.min())
print("Maximum:", test_probabilities.max())

print("\nFirst 10 test probabilities:")
print(test_probabilities[:10])

print("\nFirst 10 true labels:")
print(test_true_labels[:10])

TEST PREDICTIONS GENERATED
Number of predictions: 38052
Number of true labels: 38052

Probability range:
Minimum: 1.315385e-05
Maximum: 0.95774096

First 10 test probabilities:
[0.50625366 0.12168752 0.5937284  0.45701844 0.24997047 0.1034712
 0.24494106 0.01843453 0.01760314 0.3879419 ]

First 10 true labels:
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [61]:
# ============================================================
# FINAL TEST METRICS
# ============================================================

# Classification threshold
threshold = 0.5

# Convert probabilities into binary predictions
test_predictions = (
    test_probabilities >= threshold
).astype(int)

# ------------------------------------------------------------
# Threshold-independent metrics
# ------------------------------------------------------------

test_roc_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true_labels,
    test_probabilities
)

# ------------------------------------------------------------
# Threshold-dependent metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
)

test_precision = precision_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

test_cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("FINAL DNN TEST RESULTS")
print("=" * 60)

print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Interpretation:")
print(f"TN = {test_cm[0, 0]}")
print(f"FP = {test_cm[0, 1]}")
print(f"FN = {test_cm[1, 0]}")
print(f"TP = {test_cm[1, 1]}")

FINAL DNN TEST RESULTS
ROC-AUC   : 0.8295
PR-AUC    : 0.4232
Accuracy  : 0.7909
Precision : 0.3615
Recall    : 0.6539
F1-score  : 0.4656

Confusion Matrix:
[[26627  6123]
 [ 1835  3467]]

Confusion Matrix Interpretation:
TN = 26627
FP = 6123
FN = 1835
TP = 3467


In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UA_DNN(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(64, 32)
        self.drop3 = nn.Dropout(0.2)

        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):

        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.drop2(x)

        x = self.fc3(x)
        x = F.relu(x)
        x = self.drop3(x)

        x = self.fc4(x)

        return torch.sigmoid(x)


# Create the PRD-compliant model
model = UA_DNN(input_dim=21)

print("PRD-compliant DNN created successfully.")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

PRD-compliant DNN created successfully.
UA_DNN(
  (fc1): Linear(in_features=21, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (drop3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=32, out_features=1, bias=True)
)

Trainable parameters:
13569


In [63]:
# ============================================================
# PRD-COMPLIANT WEIGHTED BCELoss
# ============================================================

# Positive-class weight calculated from the training set
positive_weight = class_weights[1]

# Create per-sample weights
sample_weights = torch.where(
    y_train_tensor == 1.0,
    torch.tensor(positive_weight, dtype=torch.float32),
    torch.tensor(1.0, dtype=torch.float32)
)

# Weighted binary cross-entropy
criterion = nn.BCELoss(
    weight=sample_weights
)

print("PRD-compliant loss created successfully.")
print("Loss function:", criterion)
print("Positive-class weight:", positive_weight)

PRD-compliant loss created successfully.
Loss function: BCELoss()
Positive-class weight: 3.5885538760003235


In [64]:
# ============================================================
# PRD-COMPLIANT ADAM OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("PRD-compliant optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

PRD-compliant optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [65]:
# ============================================================
# FIXED PRD-COMPLIANT BCELoss
# ============================================================

criterion = nn.BCELoss(reduction='none')

positive_weight = float(class_weights[1])

print("Base BCELoss created successfully.")
print("Positive-class weight:", positive_weight)
print("Loss reduction: none")

Base BCELoss created successfully.
Positive-class weight: 3.5885538760003235
Loss reduction: none


In [66]:
# ============================================================
# PRD-COMPLIANT DNN TRAINING LOOP — FIXED WEIGHTING
# ============================================================

from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import torch


MAX_EPOCHS = 100
PATIENCE = 10

best_val_auc = -np.inf
best_model_state = None
patience_counter = 0

train_losses = []
val_losses = []
val_auc_history = []

print("Starting PRD-compliant DNN training...")
print("=" * 70)


for epoch in range(1, MAX_EPOCHS + 1):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(X_batch).squeeze(1)

        # Individual BCE losses
        individual_losses = criterion(
            predictions,
            y_batch
        )

        # Assign class weight to positive samples
        batch_weights = torch.where(
            y_batch == 1.0,
            torch.tensor(
                positive_weight,
                dtype=torch.float32
            ),
            torch.tensor(
                1.0,
                dtype=torch.float32
            )
        )

        # Apply class weights
        weighted_loss = (
            individual_losses * batch_weights
        ).mean()

        # Backpropagation
        weighted_loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate batch loss
        batch_size = X_batch.size(0)

        running_train_loss += (
            weighted_loss.item() * batch_size
        )

        train_samples += batch_size

    epoch_train_loss = (
        running_train_loss / train_samples
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    val_probabilities = []
    val_true_labels = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            # Forward pass
            predictions = model(
                X_batch
            ).squeeze(1)

            # Individual validation losses
            individual_losses = criterion(
                predictions,
                y_batch
            )

            # Same class weighting
            batch_weights = torch.where(
                y_batch == 1.0,
                torch.tensor(
                    positive_weight,
                    dtype=torch.float32
                ),
                torch.tensor(
                    1.0,
                    dtype=torch.float32
                )
            )

            weighted_val_loss = (
                individual_losses * batch_weights
            ).mean()

            # Accumulate validation loss
            batch_size = X_batch.size(0)

            running_val_loss += (
                weighted_val_loss.item() * batch_size
            )

            val_samples += batch_size

            # Store probabilities
            val_probabilities.extend(
                predictions.cpu().numpy()
            )

            val_true_labels.extend(
                y_batch.cpu().numpy()
            )


    epoch_val_loss = (
        running_val_loss / val_samples
    )


    # ========================================================
    # VALIDATION ROC-AUC
    # ========================================================

    epoch_val_auc = roc_auc_score(
        val_true_labels,
        val_probabilities
    )


    # ========================================================
    # STORE HISTORY
    # ========================================================

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    val_auc_history.append(epoch_val_auc)


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    if epoch_val_auc > best_val_auc:

        best_val_auc = epoch_val_auc

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val ROC-AUC: {epoch_val_auc:.4f} "
            f"← BEST"
        )

    else:

        patience_counter += 1

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val ROC-AUC: {epoch_val_auc:.4f} "
            f"| patience: "
            f"{patience_counter}/{PATIENCE}"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")

        print(
            f"No validation ROC-AUC improvement "
            f"for {PATIENCE} consecutive epochs."
        )

        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    print("\nBest model restored successfully.")

    print(
        f"Best validation ROC-AUC: "
        f"{best_val_auc:.4f}"
    )

print("\nTraining completed.")

Starting PRD-compliant DNN training...
Epoch 001/100 | Train Loss: 0.6863 | Val Loss: 0.6663 | Val ROC-AUC: 0.8253 ← BEST
Epoch 002/100 | Train Loss: 0.6708 | Val Loss: 0.6637 | Val ROC-AUC: 0.8274 ← BEST
Epoch 003/100 | Train Loss: 0.6689 | Val Loss: 0.6620 | Val ROC-AUC: 0.8274 ← BEST
Epoch 004/100 | Train Loss: 0.6655 | Val Loss: 0.6621 | Val ROC-AUC: 0.8276 ← BEST
Epoch 005/100 | Train Loss: 0.6654 | Val Loss: 0.6615 | Val ROC-AUC: 0.8279 ← BEST
Epoch 006/100 | Train Loss: 0.6643 | Val Loss: 0.6608 | Val ROC-AUC: 0.8283 ← BEST
Epoch 007/100 | Train Loss: 0.6634 | Val Loss: 0.6613 | Val ROC-AUC: 0.8281 | patience: 1/10
Epoch 008/100 | Train Loss: 0.6623 | Val Loss: 0.6616 | Val ROC-AUC: 0.8285 ← BEST
Epoch 009/100 | Train Loss: 0.6628 | Val Loss: 0.6623 | Val ROC-AUC: 0.8281 | patience: 1/10
Epoch 010/100 | Train Loss: 0.6623 | Val Loss: 0.6605 | Val ROC-AUC: 0.8282 | patience: 2/10
Epoch 011/100 | Train Loss: 0.6609 | Val Loss: 0.6610 | Val ROC-AUC: 0.8281 | patience: 3/10
Epoch 01

In [67]:
# ============================================================
# FINAL TEST EVALUATION — PRD-COMPLIANT DNN
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Make absolutely sure the best checkpoint is loaded
model.load_state_dict(best_model_state)

# Evaluation mode
model.eval()

test_probabilities = []
test_true_labels = []

# Generate final test probabilities
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        predictions = model(
            X_batch
        ).squeeze(1)

        test_probabilities.extend(
            predictions.cpu().numpy()
        )

        test_true_labels.extend(
            y_batch.cpu().numpy()
        )

# Convert to NumPy
test_probabilities = np.array(
    test_probabilities
)

test_true_labels = np.array(
    test_true_labels
)

# ============================================================
# CLASSIFICATION THRESHOLD
# ============================================================

threshold = 0.5

test_predictions = (
    test_probabilities >= threshold
).astype(int)

# ============================================================
# METRICS
# ============================================================

test_roc_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true_labels,
    test_probabilities
)

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
)

test_precision = precision_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

# ============================================================
# DISPLAY
# ============================================================

print("FINAL PRD-COMPLIANT DNN TEST RESULTS")
print("=" * 60)

print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Interpretation:")
print(f"TN = {test_cm[0, 0]}")
print(f"FP = {test_cm[0, 1]}")
print(f"FN = {test_cm[1, 0]}")
print(f"TP = {test_cm[1, 1]}")

print("\nProbability range:")
print(f"Minimum = {test_probabilities.min():.6f}")
print(f"Maximum = {test_probabilities.max():.6f}")

FINAL PRD-COMPLIANT DNN TEST RESULTS
ROC-AUC   : 0.8301
PR-AUC    : 0.4264
Accuracy  : 0.7872
Precision : 0.3578
Recall    : 0.6635
F1-score  : 0.4649

Confusion Matrix:
[[26437  6313]
 [ 1784  3518]]

Confusion Matrix Interpretation:
TN = 26437
FP = 6313
FN = 1784
TP = 3518

Probability range:
Minimum = 0.000108
Maximum = 0.936456


In [68]:
# ============================================================
# MC DROPOUT SETUP
# ============================================================

import torch.nn as nn

# Load the best DNN checkpoint again
model.load_state_dict(best_model_state)

# Start with the entire model in evaluation mode
model.eval()

# Activate ONLY dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

print("MC Dropout mode configured successfully.")
print("=" * 60)

# Verify the behavior of each layer
for name, module in model.named_modules():

    if isinstance(module, nn.Dropout):
        print(
            f"{name}: Dropout ACTIVE | "
            f"p = {module.p}"
        )

    elif isinstance(module, nn.BatchNorm1d):
        print(
            f"{name}: BatchNorm EVAL | "
            f"training = {module.training}"
        )

MC Dropout mode configured successfully.
bn1: BatchNorm EVAL | training = False
drop1: Dropout ACTIVE | p = 0.3
bn2: BatchNorm EVAL | training = False
drop2: Dropout ACTIVE | p = 0.3
drop3: Dropout ACTIVE | p = 0.2


In [69]:
# ============================================================
# MC DROPOUT INFERENCE — T = 50
# ============================================================

import numpy as np
import torch

# Number of stochastic forward passes
T = 50

# Make sure best checkpoint is loaded
model.load_state_dict(best_model_state)

# Evaluation mode first
model.eval()

# Activate ONLY Dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

# Store predictions from every stochastic pass
mc_predictions = []

print("Starting MC Dropout inference...")
print("=" * 60)
print(f"Number of stochastic passes (T): {T}")
print(f"Number of test samples: {len(test_true_labels)}")

# ------------------------------------------------------------
# Perform T stochastic forward passes
# ------------------------------------------------------------

with torch.no_grad():

    for t in range(T):

        pass_predictions = []

        for X_batch, _ in test_loader:

            predictions = model(
                X_batch
            ).squeeze(1)

            pass_predictions.extend(
                predictions.cpu().numpy()
            )

        pass_predictions = np.array(
            pass_predictions
        )

        mc_predictions.append(
            pass_predictions
        )

        print(
            f"Pass {t + 1:02d}/{T} completed | "
            f"Min: {pass_predictions.min():.6f} | "
            f"Max: {pass_predictions.max():.6f} | "
            f"Mean: {pass_predictions.mean():.6f}"
        )

# Convert to NumPy array
mc_predictions = np.array(
    mc_predictions
)

print("\nMC Dropout inference completed.")
print("=" * 60)

print(
    "MC prediction array shape:",
    mc_predictions.shape
)

print(
    "Expected shape:",
    (T, len(test_true_labels))
)

print(
    "Total stochastic predictions:",
    mc_predictions.size
)

Starting MC Dropout inference...
Number of stochastic passes (T): 50
Number of test samples: 38052
Pass 01/50 completed | Min: 0.000016 | Max: 0.972535 | Mean: 0.295636
Pass 02/50 completed | Min: 0.000008 | Max: 0.964224 | Mean: 0.295230
Pass 03/50 completed | Min: 0.000011 | Max: 0.969230 | Mean: 0.295780
Pass 04/50 completed | Min: 0.000002 | Max: 0.962963 | Mean: 0.295447
Pass 05/50 completed | Min: 0.000006 | Max: 0.969651 | Mean: 0.296115
Pass 06/50 completed | Min: 0.000006 | Max: 0.958834 | Mean: 0.295917
Pass 07/50 completed | Min: 0.000004 | Max: 0.977805 | Mean: 0.295860
Pass 08/50 completed | Min: 0.000019 | Max: 0.963652 | Mean: 0.295658
Pass 09/50 completed | Min: 0.000011 | Max: 0.969387 | Mean: 0.296028
Pass 10/50 completed | Min: 0.000002 | Max: 0.966035 | Mean: 0.295938
Pass 11/50 completed | Min: 0.000013 | Max: 0.987431 | Mean: 0.295665
Pass 12/50 completed | Min: 0.000011 | Max: 0.977493 | Mean: 0.296092
Pass 13/50 completed | Min: 0.000012 | Max: 0.967662 | Mean: 

In [70]:
# ============================================================
# MC DROPOUT UNCERTAINTY CALCULATION
# ============================================================

# mc_predictions shape:
# (50 stochastic passes, 38052 patients)

# ------------------------------------------------------------
# Mean prediction for every patient
# ------------------------------------------------------------

mc_mean = np.mean(
    mc_predictions,
    axis=0
)

# ------------------------------------------------------------
# Predictive variance for every patient
# ------------------------------------------------------------

mc_variance = np.var(
    mc_predictions,
    axis=0
)

# ------------------------------------------------------------
# Predictive standard deviation
# ------------------------------------------------------------

mc_std = np.sqrt(
    mc_variance
)

# ============================================================
# SANITY CHECK
# ============================================================

print("MC DROPOUT UNCERTAINTY CALCULATED")
print("=" * 60)

print("MC prediction shape:")
print(mc_predictions.shape)

print("\nMean prediction shape:")
print(mc_mean.shape)

print("\nVariance shape:")
print(mc_variance.shape)

print("\nStandard deviation shape:")
print(mc_std.shape)

print("\nExpected patient count:")
print(len(test_true_labels))

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\nMEAN PREDICTION")
print("-" * 60)
print(f"Minimum : {mc_mean.min():.6f}")
print(f"Maximum : {mc_mean.max():.6f}")
print(f"Mean    : {mc_mean.mean():.6f}")
print(f"Median  : {np.median(mc_mean):.6f}")

print("\nPREDICTIVE VARIANCE")
print("-" * 60)
print(f"Minimum : {mc_variance.min():.10f}")
print(f"Maximum : {mc_variance.max():.10f}")
print(f"Mean    : {mc_variance.mean():.10f}")
print(f"Median  : {np.median(mc_variance):.10f}")

print("\nPREDICTIVE STANDARD DEVIATION")
print("-" * 60)
print(f"Minimum : {mc_std.min():.6f}")
print(f"Maximum : {mc_std.max():.6f}")
print(f"Mean    : {mc_std.mean():.6f}")
print(f"Median  : {np.median(mc_std):.6f}")

# ============================================================
# FIRST 10 PATIENTS
# ============================================================

print("\nFIRST 10 PATIENTS")
print("-" * 60)

for i in range(10):

    print(
        f"Patient {i+1:02d} | "
        f"Mean = {mc_mean[i]:.6f} | "
        f"Variance = {mc_variance[i]:.10f} | "
        f"Std = {mc_std[i]:.6f} | "
        f"True label = {int(test_true_labels[i])}"
    )

MC DROPOUT UNCERTAINTY CALCULATED
MC prediction shape:
(50, 38052)

Mean prediction shape:
(38052,)

Variance shape:
(38052,)

Standard deviation shape:
(38052,)

Expected patient count:
38052

MEAN PREDICTION
------------------------------------------------------------
Minimum : 0.000304
Maximum : 0.924065
Mean    : 0.295743
Median  : 0.239023

PREDICTIVE VARIANCE
------------------------------------------------------------
Minimum : 0.0000002760
Maximum : 0.0145257190
Mean    : 0.0015012044
Median  : 0.0014068999

PREDICTIVE STANDARD DEVIATION
------------------------------------------------------------
Minimum : 0.000525
Maximum : 0.120523
Mean    : 0.035931
Median  : 0.037509

FIRST 10 PATIENTS
------------------------------------------------------------
Patient 01 | Mean = 0.546721 | Variance = 0.0019651442 | Std = 0.044330 | True label = 1
Patient 02 | Mean = 0.113666 | Variance = 0.0007691144 | Std = 0.027733 | True label = 0
Patient 03 | Mean = 0.525235 | Variance = 0.003919560

In [71]:
# ============================================================
# UNCERTAINTY CATEGORIZATION
# ============================================================

# Calculate percentile thresholds
uncertainty_p33 = np.percentile(
    mc_variance,
    33
)

uncertainty_p67 = np.percentile(
    mc_variance,
    67
)

# ------------------------------------------------------------
# Assign uncertainty categories
# ------------------------------------------------------------

uncertainty_levels = np.where(
    mc_variance <= uncertainty_p33,
    "Low",
    np.where(
        mc_variance <= uncertainty_p67,
        "Medium",
        "High"
    )
)

# ============================================================
# DISPLAY THRESHOLDS
# ============================================================

print("UNCERTAINTY CATEGORIZATION")
print("=" * 60)

print(
    f"33rd percentile threshold: "
    f"{uncertainty_p33:.10f}"
)

print(
    f"67th percentile threshold: "
    f"{uncertainty_p67:.10f}"
)

# ============================================================
# COUNT EACH CATEGORY
# ============================================================

unique_levels, level_counts = np.unique(
    uncertainty_levels,
    return_counts=True
)

print("\nUNCERTAINTY LEVEL DISTRIBUTION")
print("-" * 60)

for level, count in zip(
    unique_levels,
    level_counts
):

    percentage = (
        count / len(uncertainty_levels)
    ) * 100

    print(
        f"{level:8s}: "
        f"{count:6d} patients "
        f"({percentage:.2f}%)"
    )

# ============================================================
# VERIFY ALL PATIENTS WERE ASSIGNED
# ============================================================

print("\nSANITY CHECK")
print("-" * 60)

print(
    "Total patients:",
    len(uncertainty_levels)
)

print(
    "Expected patients:",
    len(test_true_labels)
)

print(
    "Missing uncertainty levels:",
    np.sum(
        pd.isna(uncertainty_levels)
    )
)

print(
    "Unique uncertainty levels:",
    np.unique(
        uncertainty_levels
    )
)

# ============================================================
# FIRST 10 PATIENTS
# ============================================================

print("\nFIRST 10 PATIENTS")
print("-" * 60)

for i in range(10):

    print(
        f"Patient {i+1:02d} | "
        f"Mean = {mc_mean[i]:.6f} | "
        f"Variance = {mc_variance[i]:.10f} | "
        f"Uncertainty = {uncertainty_levels[i]} | "
        f"True label = {int(test_true_labels[i])}"
    )

UNCERTAINTY CATEGORIZATION
33rd percentile threshold: 0.0010119812
67th percentile threshold: 0.0018078083

UNCERTAINTY LEVEL DISTRIBUTION
------------------------------------------------------------
High    :  12557 patients (33.00%)
Low     :  12557 patients (33.00%)
Medium  :  12938 patients (34.00%)

SANITY CHECK
------------------------------------------------------------
Total patients: 38052
Expected patients: 38052
Missing uncertainty levels: 0
Unique uncertainty levels: ['High' 'Low' 'Medium']

FIRST 10 PATIENTS
------------------------------------------------------------
Patient 01 | Mean = 0.546721 | Variance = 0.0019651442 | Uncertainty = High | True label = 1
Patient 02 | Mean = 0.113666 | Variance = 0.0007691144 | Uncertainty = Low | True label = 0
Patient 03 | Mean = 0.525235 | Variance = 0.0039195600 | Uncertainty = High | True label = 0
Patient 04 | Mean = 0.512595 | Variance = 0.0022348210 | Uncertainty = High | True label = 0
Patient 05 | Mean = 0.256157 | Variance =

In [72]:
# ============================================================
# UNCERTAINTY GROUP VALIDATION
# ============================================================

print("UNCERTAINTY GROUP VALIDATION")
print("=" * 70)

for level in ["Low", "Medium", "High"]:

    # Select patients belonging to this uncertainty level
    mask = uncertainty_levels == level

    # Extract their variances
    group_variance = mc_variance[mask]

    # Extract their standard deviations
    group_std = mc_std[mask]

    # Extract their mean predictions
    group_mean = mc_mean[mask]

    print(f"\n{level.upper()} UNCERTAINTY")
    print("-" * 70)

    print(f"Number of patients : {len(group_variance):,}")

    print(
        f"Mean variance      : "
        f"{np.mean(group_variance):.10f}"
    )

    print(
        f"Median variance    : "
        f"{np.median(group_variance):.10f}"
    )

    print(
        f"Mean std deviation : "
        f"{np.mean(group_std):.6f}"
    )

    print(
        f"Mean prediction    : "
        f"{np.mean(group_mean):.6f}"
    )

# ============================================================
# ORDERING CHECK
# ============================================================

low_mean_variance = np.mean(
    mc_variance[uncertainty_levels == "Low"]
)

medium_mean_variance = np.mean(
    mc_variance[uncertainty_levels == "Medium"]
)

high_mean_variance = np.mean(
    mc_variance[uncertainty_levels == "High"]
)

print("\n" + "=" * 70)
print("ORDERING CHECK")
print("=" * 70)

print(
    f"Low    mean variance    : "
    f"{low_mean_variance:.10f}"
)

print(
    f"Medium mean variance    : "
    f"{medium_mean_variance:.10f}"
)

print(
    f"High   mean variance    : "
    f"{high_mean_variance:.10f}"
)

print("\nExpected relationship:")
print("Low < Medium < High")

if (
    low_mean_variance
    < medium_mean_variance
    < high_mean_variance
):
    print("\n✓ UNCERTAINTY ORDERING VERIFIED")
else:
    print("\n✗ WARNING: UNCERTAINTY ORDERING NOT VERIFIED")

UNCERTAINTY GROUP VALIDATION

LOW UNCERTAINTY
----------------------------------------------------------------------
Number of patients : 12,557
Mean variance      : 0.0004525014
Median variance    : 0.0004086724
Mean std deviation : 0.019519
Mean prediction    : 0.080018

MEDIUM UNCERTAINTY
----------------------------------------------------------------------
Number of patients : 12,938
Mean variance      : 0.0014061445
Median variance    : 0.0014068999
Mean std deviation : 0.037380
Mean prediction    : 0.330575

HIGH UNCERTAINTY
----------------------------------------------------------------------
Number of patients : 12,557
Mean variance      : 0.0026478516
Median variance    : 0.0023815674
Mean std deviation : 0.050850
Mean prediction    : 0.475580

ORDERING CHECK
Low    mean variance    : 0.0004525014
Medium mean variance    : 0.0014061445
High   mean variance    : 0.0026478516

Expected relationship:
Low < Medium < High

✓ UNCERTAINTY ORDERING VERIFIED


In [74]:
# ============================================================
# FIND EXISTING TRAINING DATA VARIABLES
# ============================================================

print("Variables containing 'train':")
print("=" * 60)

for name in dir():
    if "train" in name.lower():
        try:
            obj = globals()[name]
            print(
                f"{name:30s} | "
                f"type = {type(obj).__name__}"
            )

            if hasattr(obj, "shape"):
                print(
                    f"{'':30s} | "
                    f"shape = {obj.shape}"
                )
        except:
            pass

Variables containing 'train':
X_train                        | type = DataFrame
                               | shape = (177576, 21)
X_train_np                     | type = ndarray
                               | shape = (177576, 21)
X_train_processed              | type = DataFrame
                               | shape = (177576, 21)
X_train_tensor                 | type = Tensor
                               | shape = torch.Size([177576, 21])
epoch_train_loss               | type = float
running_train_loss             | type = float
train_dataset                  | type = TensorDataset
train_loader                   | type = DataLoader
train_losses                   | type = list
train_samples                  | type = int
train_test_split               | type = function
y_train                        | type = Series
                               | shape = (177576,)
y_train_processed              | type = ndarray
                               | shape = (177576,)
y_train_tensor 

In [75]:
# ============================================================
# PHASE 6 — LIME EXPLAINER SETUP
# ============================================================

from lime.lime_tabular import LimeTabularExplainer

# Feature names
feature_names = list(X_train_processed.columns)

print("LIME EXPLAINER SETUP")
print("=" * 70)

print(f"Number of features: {len(feature_names)}")

print("\nFeature names:")
for i, feature in enumerate(feature_names):
    print(f"{i:02d} | {feature}")


# ------------------------------------------------------------
# LIME prediction function
# ------------------------------------------------------------

def lime_predict_proba(data):

    """
    Receives NumPy feature arrays from LIME
    and returns probabilities for both classes.

    Column 0 = No Diabetes
    Column 1 = Diabetes
    """

    # Convert NumPy data to PyTorch tensor
    tensor_data = torch.tensor(
        data,
        dtype=torch.float32
    )

    # Evaluation mode for normal DNN prediction
    model.eval()

    # No gradients needed
    with torch.no_grad():

        diabetes_probability = (
            model(tensor_data)
            .squeeze(1)
            .cpu()
            .numpy()
        )

    # Probability of no diabetes
    no_diabetes_probability = (
        1.0 - diabetes_probability
    )

    # LIME expects probability for both classes
    probabilities = np.column_stack([
        no_diabetes_probability,
        diabetes_probability
    ])

    return probabilities


# ------------------------------------------------------------
# Create LIME explainer
# ------------------------------------------------------------

lime_explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=feature_names,
    class_names=[
        "No Diabetes",
        "Diabetes"
    ],
    mode="classification",
    discretize_continuous=False,
    random_state=42
)

print("\n" + "=" * 70)
print("LIME EXPLAINER CREATED SUCCESSFULLY")
print("=" * 70)

print("Training data shape:", X_train_np.shape)
print("Number of features:", len(feature_names))
print("Classes:", ["No Diabetes", "Diabetes"])
print("Mode: classification")
print("Discretize continuous: False")
print("Random state: 42")

LIME EXPLAINER SETUP
Number of features: 21

Feature names:
00 | HighBP
01 | HighChol
02 | CholCheck
03 | BMI
04 | Smoker
05 | Stroke
06 | HeartDiseaseorAttack
07 | PhysActivity
08 | Fruits
09 | Veggies
10 | HvyAlcoholConsump
11 | AnyHealthcare
12 | NoDocbcCost
13 | GenHlth
14 | MentHlth
15 | PhysHlth
16 | DiffWalk
17 | Sex
18 | Age
19 | Education
20 | Income

LIME EXPLAINER CREATED SUCCESSFULLY
Training data shape: (177576, 21)
Number of features: 21
Classes: ['No Diabetes', 'Diabetes']
Mode: classification
Discretize continuous: False
Random state: 42


In [76]:
# ============================================================
# PHASE 6 — STEP 2
# GENERATE FIRST LIME EXPLANATION
# ============================================================

# Select Patient 1 from the test set
patient_index = 0

patient_features = X_test_np[patient_index]

print("GENERATING LIME EXPLANATION")
print("=" * 70)

print(f"Patient index: {patient_index}")
print(f"Number of features: {len(patient_features)}")

# ------------------------------------------------------------
# Generate LIME explanation
# ------------------------------------------------------------

lime_explanation = lime_explainer.explain_instance(
    patient_features,
    lime_predict_proba,
    num_features=21,
    top_labels=1,
    num_samples=5000
)

# ------------------------------------------------------------
# Determine predicted class
# ------------------------------------------------------------

patient_probability = mc_mean[patient_index]

predicted_class = (
    "Diabetes"
    if patient_probability >= 0.5
    else "No Diabetes"
)

# ------------------------------------------------------------
# Display prediction information
# ------------------------------------------------------------

print("\nPATIENT PREDICTION")
print("-" * 70)

print(
    f"MC Mean Probability : "
    f"{patient_probability:.6f}"
)

print(
    f"Predicted Class     : "
    f"{predicted_class}"
)

print(
    f"True Label          : "
    f"{int(test_true_labels[patient_index])}"
)

print(
    f"Predictive Variance : "
    f"{mc_variance[patient_index]:.10f}"
)

print(
    f"Uncertainty Level   : "
    f"{uncertainty_levels[patient_index]}"
)

# ------------------------------------------------------------
# Extract LIME feature contributions
# ------------------------------------------------------------

lime_features = lime_explanation.as_list(
    label=1
)

print("\nLIME FEATURE CONTRIBUTIONS")
print("-" * 70)

for feature, contribution in lime_features:

    direction = (
        "toward Diabetes"
        if contribution > 0
        else "toward No Diabetes"
    )

    print(
        f"{feature:30s} | "
        f"{contribution:+.6f} | "
        f"{direction}"
    )

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\nLIME VALIDATION")
print("-" * 70)

print(
    "Number of explanation features:",
    len(lime_features)
)

print(
    "Expected maximum features:",
    21
)

print(
    "Explanation generated successfully:",
    len(lime_features) > 0
)

GENERATING LIME EXPLANATION
Patient index: 0
Number of features: 21

PATIENT PREDICTION
----------------------------------------------------------------------
MC Mean Probability : 0.546721
Predicted Class     : Diabetes
True Label          : 1
Predictive Variance : 0.0019651442
Uncertainty Level   : High

LIME FEATURE CONTRIBUTIONS
----------------------------------------------------------------------
GenHlth                        | +0.106410 | toward Diabetes
BMI                            | +0.100567 | toward Diabetes
Age                            | +0.076480 | toward Diabetes
HighBP                         | +0.061613 | toward Diabetes
HighChol                       | +0.054754 | toward Diabetes
CholCheck                      | +0.040783 | toward Diabetes
HeartDiseaseorAttack           | +0.029638 | toward Diabetes
Income                         | -0.029124 | toward No Diabetes
HvyAlcoholConsump              | -0.026854 | toward No Diabetes
Sex                            | +0.020

In [77]:
# ============================================================
# STEP 1A — Uncertainty Stratification
# ============================================================

# PRD-defined uncertainty thresholds
LOW_THRESHOLD = 0.05
HIGH_THRESHOLD = 0.15

# Assign each test instance to an uncertainty stratum
uncertainty_stratum = np.where(
    mc_variance < LOW_THRESHOLD,
    "LOW",
    np.where(
        mc_variance < HIGH_THRESHOLD,
        "MEDIUM",
        "HIGH"
    )
)

# Count instances in each stratum
stratum_counts = pd.Series(uncertainty_stratum).value_counts()

# Ensure consistent ordering
stratum_counts = stratum_counts.reindex(
    ["LOW", "MEDIUM", "HIGH"],
    fill_value=0
)

print("PRD Uncertainty Stratification")
print("=" * 40)
print(f"LOW    (σ² < 0.05):       {stratum_counts['LOW']:,}")
print(f"MEDIUM (0.05 ≤ σ² < 0.15): {stratum_counts['MEDIUM']:,}")
print(f"HIGH   (σ² ≥ 0.15):       {stratum_counts['HIGH']:,}")
print("=" * 40)
print(f"Total:                    {stratum_counts.sum():,}")

PRD Uncertainty Stratification
LOW    (σ² < 0.05):       38,052
MEDIUM (0.05 ≤ σ² < 0.15): 0
HIGH   (σ² ≥ 0.15):       0
Total:                    38,052


In [81]:
# ============================================================
# STEP 1B — Adjusted Uncertainty Threshold Check
# ============================================================

ADJUSTED_LOW_THRESHOLD = 0.03
ADJUSTED_HIGH_THRESHOLD = 0.10

adjusted_stratum = np.where(
    mc_variance < ADJUSTED_LOW_THRESHOLD,
    "LOW",
    np.where(
        mc_variance < ADJUSTED_HIGH_THRESHOLD,
        "MEDIUM",
        "HIGH"
    )
)

adjusted_counts = pd.Series(adjusted_stratum).value_counts()

adjusted_counts = adjusted_counts.reindex(
    ["LOW", "MEDIUM", "HIGH"],
    fill_value=0
)

print("Adjusted PRD Uncertainty Stratification")
print("=" * 45)
print(f"LOW    (σ² < 0.03):          {adjusted_counts['LOW']:,}")
print(f"MEDIUM (0.03 ≤ σ² < 0.10): {adjusted_counts['MEDIUM']:,}")
print(f"HIGH   (σ² ≥ 0.10):         {adjusted_counts['HIGH']:,}")
print("=" * 45)
print(f"Total:                       {adjusted_counts.sum():,}")

print("\nMinimum stratum requirement: 150 instances")
print(f"LOW ≥ 150:    {adjusted_counts['LOW'] >= 150}")
print(f"MEDIUM ≥ 150: {adjusted_counts['MEDIUM'] >= 150}")
print(f"HIGH ≥ 150:   {adjusted_counts['HIGH'] >= 150}")

Adjusted PRD Uncertainty Stratification
LOW    (σ² < 0.03):          38,052
MEDIUM (0.03 ≤ σ² < 0.10): 0
HIGH   (σ² ≥ 0.10):         0
Total:                       38,052

Minimum stratum requirement: 150 instances
LOW ≥ 150:    True
MEDIUM ≥ 150: False
HIGH ≥ 150:   False


In [82]:
# ============================================================
# STEP 1C — Full MC Dropout Variance Distribution
# ============================================================

variance_percentiles = {
    "50th (Median)": np.percentile(mc_variance, 50),
    "90th": np.percentile(mc_variance, 90),
    "95th": np.percentile(mc_variance, 95),
    "99th": np.percentile(mc_variance, 99),
    "99.9th": np.percentile(mc_variance, 99.9),
    "Maximum": np.max(mc_variance)
}

print("MC Dropout Predictive Variance Distribution")
print("=" * 50)

for percentile, value in variance_percentiles.items():
    print(f"{percentile:<18}: {value:.10f}")

print("=" * 50)

# Additional verification of the MC inference state
print("\nMC Dropout Inference Configuration")
print("=" * 50)

print(f"Number of MC passes (T): {mc_predictions.shape[0]}")
print(f"Number of test instances: {mc_predictions.shape[1]}")

print("\nCurrent model state:")
print(f"model.training = {model.training}")

print("\nDropout layer states:")
print(f"drop1.training = {model.drop1.training}")
print(f"drop2.training = {model.drop2.training}")
print(f"drop3.training = {model.drop3.training}")

print("\nBatchNorm layer states:")
print(f"bn1.training = {model.bn1.training}")
print(f"bn2.training = {model.bn2.training}")

MC Dropout Predictive Variance Distribution
50th (Median)     : 0.0014068999
90th              : 0.0027569428
95th              : 0.0033079339
99th              : 0.0048498134
99.9th            : 0.0085418466
Maximum           : 0.0145257190

MC Dropout Inference Configuration
Number of MC passes (T): 50
Number of test instances: 38052

Current model state:
model.training = False

Dropout layer states:
drop1.training = False
drop2.training = False
drop3.training = False

BatchNorm layer states:
bn1.training = False
bn2.training = False


In [85]:
# ============================================================
# STEP 1C — Correct MC Dropout Inference
# ============================================================

T = 50

# Activate training mode so Dropout layers are ACTIVE
model.train()

mc_predictions_corrected = []

with torch.no_grad():
    for _ in range(T):
        batch_predictions = []

        for X_batch, _ in test_loader:
            predictions = model(X_batch)
            batch_predictions.append(predictions.squeeze(1).cpu())

        pass_predictions = torch.cat(batch_predictions)
        mc_predictions_corrected.append(pass_predictions)

# Convert to tensor
mc_predictions_corrected = torch.stack(mc_predictions_corrected)

# Calculate mean prediction and predictive variance
mc_mean_corrected = mc_predictions_corrected.mean(dim=0).numpy()
mc_variance_corrected = mc_predictions_corrected.var(dim=0).numpy()
mc_std_corrected = np.sqrt(mc_variance_corrected)

print("Corrected MC Dropout Inference")
print("=" * 50)
print(f"MC passes (T): {T}")
print(f"Prediction shape: {mc_predictions_corrected.shape}")
print(f"Mean prediction shape: {mc_mean_corrected.shape}")
print(f"Variance shape: {mc_variance_corrected.shape}")
print("=" * 50)

# ------------------------------------------------------------
# Verify Dropout is actually active
# ------------------------------------------------------------

print("\nModel/Dropout State")
print("=" * 50)
print(f"model.training = {model.training}")
print(f"drop1.training = {model.drop1.training}")
print(f"drop2.training = {model.drop2.training}")
print(f"drop3.training = {model.drop3.training}")
print(f"bn1.training = {model.bn1.training}")
print(f"bn2.training = {model.bn2.training}")

# ------------------------------------------------------------
# Variance distribution
# ------------------------------------------------------------

print("\nMC Dropout Variance Distribution")
print("=" * 50)

print(f"50th percentile (median): {np.percentile(mc_variance_corrected, 50):.10f}")
print(f"90th percentile:          {np.percentile(mc_variance_corrected, 90):.10f}")
print(f"95th percentile:          {np.percentile(mc_variance_corrected, 95):.10f}")
print(f"99th percentile:          {np.percentile(mc_variance_corrected, 99):.10f}")
print(f"99.9th percentile:        {np.percentile(mc_variance_corrected, 99.9):.10f}")
print(f"Maximum:                  {np.max(mc_variance_corrected):.10f}")

# ------------------------------------------------------------
# Test ROC-AUC using MC mean probability
# ------------------------------------------------------------

mc_auc_corrected = roc_auc_score(
    y_test_processed,
    mc_mean_corrected
)

print("\nTest ROC-AUC using MC Mean Probability")
print("=" * 50)
print(f"ROC-AUC: {mc_auc_corrected:.4f}")

Corrected MC Dropout Inference
MC passes (T): 50
Prediction shape: torch.Size([50, 38052])
Mean prediction shape: (38052,)
Variance shape: (38052,)

Model/Dropout State
model.training = True
drop1.training = True
drop2.training = True
drop3.training = True
bn1.training = True
bn2.training = True

MC Dropout Variance Distribution
50th percentile (median): 0.0014238660
90th percentile:          0.0028060151
95th percentile:          0.0033530323
99th percentile:          0.0048243385
99.9th percentile:        0.0083678355
Maximum:                  0.0168061405

Test ROC-AUC using MC Mean Probability
ROC-AUC: 0.8293


In [86]:
# ============================================================
# STEP 1D — MC Dropout with BatchNorm Kept in Evaluation Mode
# ============================================================

T = 50

# Start from evaluation mode
# This keeps BatchNorm fixed using the statistics learned during training.
model.eval()

# Activate ONLY Dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

# Verify states before inference
print("MC Dropout Inference State")
print("=" * 50)
print(f"model.training = {model.training}")
print(f"drop1.training = {model.drop1.training}")
print(f"drop2.training = {model.drop2.training}")
print(f"drop3.training = {model.drop3.training}")
print(f"bn1.training = {model.bn1.training}")
print(f"bn2.training = {model.bn2.training}")

# ------------------------------------------------------------
# Run T = 50 stochastic forward passes
# ------------------------------------------------------------

mc_predictions_final = []

with torch.no_grad():
    for _ in range(T):

        batch_predictions = []

        for X_batch, _ in test_loader:
            predictions = model(X_batch)
            batch_predictions.append(
                predictions.squeeze(1).cpu()
            )

        pass_predictions = torch.cat(batch_predictions)
        mc_predictions_final.append(pass_predictions)

# Stack predictions
mc_predictions_final = torch.stack(mc_predictions_final)

# ------------------------------------------------------------
# Calculate MC mean and predictive variance
# ------------------------------------------------------------

mc_mean_final = mc_predictions_final.mean(dim=0).numpy()

mc_variance_final = mc_predictions_final.var(dim=0).numpy()

mc_std_final = np.sqrt(mc_variance_final)

print("\nMC Prediction Results")
print("=" * 50)
print(f"Prediction shape: {mc_predictions_final.shape}")
print(f"MC mean shape:    {mc_mean_final.shape}")
print(f"Variance shape:   {mc_variance_final.shape}")

# ------------------------------------------------------------
# Variance distribution
# ------------------------------------------------------------

print("\nMC Dropout Variance Distribution")
print("=" * 50)

print(
    f"50th percentile (median): "
    f"{np.percentile(mc_variance_final, 50):.10f}"
)

print(
    f"90th percentile:          "
    f"{np.percentile(mc_variance_final, 90):.10f}"
)

print(
    f"95th percentile:          "
    f"{np.percentile(mc_variance_final, 95):.10f}"
)

print(
    f"99th percentile:          "
    f"{np.percentile(mc_variance_final, 99):.10f}"
)

print(
    f"99.9th percentile:        "
    f"{np.percentile(mc_variance_final, 99.9):.10f}"
)

print(
    f"Maximum:                  "
    f"{np.max(mc_variance_final):.10f}"
)

# ------------------------------------------------------------
# Test ROC-AUC using MC mean probability
# ------------------------------------------------------------

mc_auc_final = roc_auc_score(
    y_test_processed,
    mc_mean_final
)

print("\nTest ROC-AUC using MC Mean Probability")
print("=" * 50)
print(f"ROC-AUC: {mc_auc_final:.4f}")

MC Dropout Inference State
model.training = False
drop1.training = True
drop2.training = True
drop3.training = True
bn1.training = False
bn2.training = False

MC Prediction Results
Prediction shape: torch.Size([50, 38052])
MC mean shape:    (38052,)
Variance shape:   (38052,)

MC Dropout Variance Distribution
50th percentile (median): 0.0013986706
90th percentile:          0.0027749955
95th percentile:          0.0033128543
99th percentile:          0.0047697742
99.9th percentile:        0.0086126337
Maximum:                  0.0175067578

Test ROC-AUC using MC Mean Probability
ROC-AUC: 0.8303


In [87]:
# ============================================================
# STEP 1E — Final PRD Threshold Verification
# ============================================================

threshold_sets = {
    "Original PRD (0.05, 0.15)": (0.05, 0.15),
    "Adjusted PRD (0.03, 0.10)": (0.03, 0.10)
}

print("FINAL UNCERTAINTY THRESHOLD VERIFICATION")
print("=" * 65)

for name, (low_threshold, high_threshold) in threshold_sets.items():

    strata = np.where(
        mc_variance_final < low_threshold,
        "LOW",
        np.where(
            mc_variance_final < high_threshold,
            "MEDIUM",
            "HIGH"
        )
    )

    counts = pd.Series(strata).value_counts()
    counts = counts.reindex(
        ["LOW", "MEDIUM", "HIGH"],
        fill_value=0
    )

    print(f"\n{name}")
    print("-" * 65)
    print(f"LOW    (< {low_threshold}):       {counts['LOW']:,}")
    print(f"MEDIUM ({low_threshold}–<{high_threshold}): {counts['MEDIUM']:,}")
    print(f"HIGH   (≥ {high_threshold}):      {counts['HIGH']:,}")

    print(
        f"Minimum ≥150 check: "
        f"LOW={counts['LOW'] >= 150}, "
        f"MEDIUM={counts['MEDIUM'] >= 150}, "
        f"HIGH={counts['HIGH'] >= 150}"
    )

print("\nFinal observed variance range")
print("-" * 65)
print(f"Minimum σ²: {np.min(mc_variance_final):.10f}")
print(f"Maximum σ²: {np.max(mc_variance_final):.10f}")

FINAL UNCERTAINTY THRESHOLD VERIFICATION

Original PRD (0.05, 0.15)
-----------------------------------------------------------------
LOW    (< 0.05):       38,052
MEDIUM (0.05–<0.15): 0
HIGH   (≥ 0.15):      0
Minimum ≥150 check: LOW=True, MEDIUM=False, HIGH=False

Adjusted PRD (0.03, 0.10)
-----------------------------------------------------------------
LOW    (< 0.03):       38,052
MEDIUM (0.03–<0.1): 0
HIGH   (≥ 0.1):      0
Minimum ≥150 check: LOW=True, MEDIUM=False, HIGH=False

Final observed variance range
-----------------------------------------------------------------
Minimum σ²: 0.0000003084
Maximum σ²: 0.0175067578


In [88]:
# ============================================================
# STEP 1F — Variance Distribution Diagnostic
# ============================================================

print("MC DROPOUT VARIANCE DIAGNOSTIC")
print("=" * 60)

# Count how many observations exceed important variance levels
diagnostic_thresholds = [0.001, 0.002, 0.003, 0.005, 0.01, 0.015, 0.02]

for threshold in diagnostic_thresholds:
    count = np.sum(mc_variance_final >= threshold)
    percentage = (count / len(mc_variance_final)) * 100

    print(
        f"σ² ≥ {threshold:<6}: "
        f"{count:>6,} instances "
        f"({percentage:>6.2f}%)"
    )

print("=" * 60)

# Number of unique variance values
print(f"Total variance observations: {len(mc_variance_final):,}")
print(
    f"Instances with σ² > 0.01: "
    f"{np.sum(mc_variance_final > 0.01):,}"
)

print(
    f"Instances with σ² > 0.015: "
    f"{np.sum(mc_variance_final > 0.015):,}"
)

MC DROPOUT VARIANCE DIAGNOSTIC
σ² ≥ 0.001 : 25,834 instances ( 67.89%)
σ² ≥ 0.002 :  9,985 instances ( 26.24%)
σ² ≥ 0.003 :  2,894 instances (  7.61%)
σ² ≥ 0.005 :    313 instances (  0.82%)
σ² ≥ 0.01  :     21 instances (  0.06%)
σ² ≥ 0.015 :      3 instances (  0.01%)
σ² ≥ 0.02  :      0 instances (  0.00%)
Total variance observations: 38,052
Instances with σ² > 0.01: 21
Instances with σ² > 0.015: 3


In [89]:
# ============================================================
# STEP 1 — Final Percentile-Based Uncertainty Stratification
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Compute percentile boundaries
# ------------------------------------------------------------

p33 = np.percentile(mc_variance_final, 33)
p66 = np.percentile(mc_variance_final, 66)

print("Percentile-Based Uncertainty Stratification")
print("=" * 65)
print(f"33rd percentile σ²: {p33:.10f}")
print(f"66th percentile σ²: {p66:.10f}")
print()

# ------------------------------------------------------------
# 2. Rank all instances by predictive variance
# ------------------------------------------------------------

n_instances = len(mc_variance_final)

sorted_indices = np.argsort(mc_variance_final)

# Each stratum contains exactly one-third of the test set
stratum_size = n_instances // 3

low_indices = sorted_indices[:stratum_size]
medium_indices = sorted_indices[stratum_size:2 * stratum_size]
high_indices = sorted_indices[2 * stratum_size:]

# ------------------------------------------------------------
# 3. Create stratum labels
# ------------------------------------------------------------

final_stratum = np.empty(n_instances, dtype=object)

final_stratum[low_indices] = "LOW"
final_stratum[medium_indices] = "MEDIUM"
final_stratum[high_indices] = "HIGH"

# ------------------------------------------------------------
# 4. Create diagnostic DataFrame
# ------------------------------------------------------------

stratification_df = pd.DataFrame({
    "test_instance_id": np.arange(n_instances),
    "sigma_squared": mc_variance_final,
    "uncertainty_stratum": final_stratum
})

# ------------------------------------------------------------
# 5. Report stratum sizes
# ------------------------------------------------------------

stratum_counts = (
    stratification_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

print("Stratum Sizes")
print("-" * 65)

for stratum in ["LOW", "MEDIUM", "HIGH"]:
    print(f"{stratum:<10}: {stratum_counts[stratum]:,}")

print("-" * 65)
print(f"Total     : {stratum_counts.sum():,}")

# ------------------------------------------------------------
# 6. Report actual σ² range within each stratum
# ------------------------------------------------------------

print("\nσ² Range Within Each Stratum")
print("-" * 65)

for stratum in ["LOW", "MEDIUM", "HIGH"]:

    values = stratification_df.loc[
        stratification_df["uncertainty_stratum"] == stratum,
        "sigma_squared"
    ]

    print(
        f"{stratum:<10}: "
        f"min = {values.min():.10f}, "
        f"max = {values.max():.10f}"
    )

# ------------------------------------------------------------
# 7. Verify the 150-instance minimum requirement
# ------------------------------------------------------------

print("\nMinimum Stratum Requirement")
print("-" * 65)

for stratum in ["LOW", "MEDIUM", "HIGH"]:
    print(
        f"{stratum:<10} ≥ 150: "
        f"{stratum_counts[stratum] >= 150}"
    )

# ------------------------------------------------------------
# 8. Verify exact equal-sized strata
# ------------------------------------------------------------

expected_size = n_instances // 3

print("\nEqual-Stratum Verification")
print("-" * 65)
print(f"Expected per stratum: {expected_size:,}")
print(
    f"All strata exactly equal: "
    f"{all(stratum_counts == expected_size)}"
)

# ------------------------------------------------------------
# 9. Sample 200 instances from each stratum
# ------------------------------------------------------------

RANDOM_STATE = 42
SAMPLE_SIZE = 200

sampled_parts = []

for stratum in ["LOW", "MEDIUM", "HIGH"]:

    stratum_data = stratification_df[
        stratification_df["uncertainty_stratum"] == stratum
    ]

    sampled = stratum_data.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )

    sampled_parts.append(sampled)

stratified_samples = (
    pd.concat(sampled_parts)
    .sort_values(
        ["uncertainty_stratum", "test_instance_id"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 10. Verify final sample
# ------------------------------------------------------------

print("\nFinal XAI Sample")
print("=" * 65)
print(f"Total sampled instances: {len(stratified_samples):,}")

print("\nSamples per stratum:")

sample_counts = (
    stratified_samples["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

for stratum in ["LOW", "MEDIUM", "HIGH"]:
    print(f"{stratum:<10}: {sample_counts[stratum]:,}")

# ------------------------------------------------------------
# 11. Save the 600-instance stratified sample
# ------------------------------------------------------------

output_file = "stratified_samples.csv"

stratified_samples.to_csv(
    output_file,
    index=False
)

print("\nSaved successfully:")
print(output_file)

Percentile-Based Uncertainty Stratification
33rd percentile σ²: 0.0010218532
66th percentile σ²: 0.0017743841

Stratum Sizes
-----------------------------------------------------------------
LOW       : 12,684
MEDIUM    : 12,684
HIGH      : 12,684
-----------------------------------------------------------------
Total     : 38,052

σ² Range Within Each Stratum
-----------------------------------------------------------------
LOW       : min = 0.0000003084, max = 0.0010297309
MEDIUM    : min = 0.0010297557, max = 0.0017923317
HIGH      : min = 0.0017924170, max = 0.0175067578

Minimum Stratum Requirement
-----------------------------------------------------------------
LOW        ≥ 150: True
MEDIUM     ≥ 150: True
HIGH       ≥ 150: True

Equal-Stratum Verification
-----------------------------------------------------------------
Expected per stratum: 12,684
All strata exactly equal: True

Final XAI Sample
Total sampled instances: 600

Samples per stratum:
LOW       : 200
MEDIUM    : 200

In [91]:
# ============================================================
# STEP 2A — Load and Verify SHAP Sample
# ============================================================

# Load the 600-instance stratified sample
shap_sample_df = pd.read_csv("stratified_samples.csv")

print("SHAP Sample Verification")
print("=" * 60)

# Basic shape
print(f"Sample shape: {shap_sample_df.shape}")

# Required columns
print(f"\nColumns:")
print(shap_sample_df.columns.tolist())

# Stratum counts
print("\nStratum counts:")
print(
    shap_sample_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

# Check unique instance IDs
print(f"\nUnique test instance IDs: "
      f"{shap_sample_df['test_instance_id'].nunique()}")

# Check ID range
print(f"Minimum test instance ID: "
      f"{shap_sample_df['test_instance_id'].min()}")

print(f"Maximum test instance ID: "
      f"{shap_sample_df['test_instance_id'].max()}")

# Check whether every ID is valid for the test set
valid_ids = (
    (shap_sample_df["test_instance_id"] >= 0) &
    (shap_sample_df["test_instance_id"] < len(X_test_processed))
)

print(f"\nAll instance IDs valid: {valid_ids.all()}")

# Check expected sample size
print(f"Exactly 600 instances: "
      f"{len(shap_sample_df) == 600}")

# Check exactly 200 per stratum
stratum_counts = (
    shap_sample_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"], fill_value=0)
)

print(
    f"Exactly 200 per stratum: "
    f"{all(stratum_counts == 200)}"
)

# ------------------------------------------------------------
# Retrieve the corresponding feature rows
# ------------------------------------------------------------

shap_instance_ids = shap_sample_df["test_instance_id"].to_numpy()

X_shap = X_test_processed.iloc[shap_instance_ids].copy()
y_shap = y_test_processed[shap_instance_ids]
print("\nSHAP Input Data")
print("=" * 60)
print(f"X_shap shape: {X_shap.shape}")
print(f"y_shap shape: {y_shap.shape}")

print("\nFirst 5 instance IDs:")
print(shap_instance_ids[:5])

print("\nFirst 5 true labels:")
print(y_shap[:5])

print("\nFeature count:")
print(f"{X_shap.shape[1]} features")

SHAP Sample Verification
Sample shape: (600, 3)

Columns:
['test_instance_id', 'sigma_squared', 'uncertainty_stratum']

Stratum counts:
uncertainty_stratum
LOW       200
MEDIUM    200
HIGH      200
Name: count, dtype: int64

Unique test instance IDs: 600
Minimum test instance ID: 481
Maximum test instance ID: 37842

All instance IDs valid: True
Exactly 600 instances: True
Exactly 200 per stratum: True

SHAP Input Data
X_shap shape: (600, 21)
y_shap shape: (600,)

First 5 instance IDs:
[ 529  850  905 1029 1041]

First 5 true labels:
[0. 0. 0. 1. 1.]

Feature count:
21 features


In [92]:
# Retrieve labels for the 600 SHAP instances
y_shap = y_test_processed[shap_instance_ids]

print("\nSHAP Input Data")
print("=" * 60)
print(f"X_shap shape: {X_shap.shape}")
print(f"y_shap shape: {y_shap.shape}")

print("\nFirst 5 instance IDs:")
print(shap_instance_ids[:5])

print("\nFirst 5 true labels:")
print(y_shap[:5])

print("\nFeature count:")
print(f"{X_shap.shape[1]} features")


SHAP Input Data
X_shap shape: (600, 21)
y_shap shape: (600,)

First 5 instance IDs:
[ 529  850  905 1029 1041]

First 5 true labels:
[0. 0. 0. 1. 1.]

Feature count:
21 features


In [93]:
# ============================================================
# STEP 2B — SHAP Explainer Setup
# ============================================================

import shap
import numpy as np
import torch

# ------------------------------------------------------------
# SHAP prediction function
# ------------------------------------------------------------

def shap_predict(data):
    """
    SHAP-compatible prediction function.

    Input:
        data -> NumPy array of shape (n_instances, 21)

    Output:
        Diabetes probability for each instance.
    """

    tensor_data = torch.tensor(
        data,
        dtype=torch.float32
    )

    model.eval()

    with torch.no_grad():
        predictions = model(tensor_data).squeeze(1).cpu().numpy()

    return predictions


# ------------------------------------------------------------
# Create SHAP background dataset
# ------------------------------------------------------------

SHAP_BACKGROUND_SIZE = 500

np.random.seed(42)

background_indices = np.random.choice(
    len(X_train_processed),
    size=SHAP_BACKGROUND_SIZE,
    replace=False
)

X_shap_background = X_train_processed.iloc[
    background_indices
].to_numpy(dtype=np.float32)

print("SHAP Background Dataset")
print("=" * 60)
print(f"Background shape: {X_shap_background.shape}")

# ------------------------------------------------------------
# Create SHAP explainer
# ------------------------------------------------------------

shap_explainer = shap.Explainer(
    shap_predict,
    X_shap_background,
    feature_names=X_shap.columns.tolist()
)

print("\nSHAP Explainer")
print("=" * 60)
print(f"Explainer type: {type(shap_explainer).__name__}")
print("Explainer created successfully.")

SHAP Background Dataset
Background shape: (500, 21)

SHAP Explainer
Explainer type: PermutationExplainer
Explainer created successfully.


In [94]:
# ============================================================
# STEP 2C — SHAP Single-Instance Verification
# ============================================================

# Explain the first sampled test instance
X_single_shap = X_shap.iloc[[0]].to_numpy(dtype=np.float32)

# Generate SHAP explanation
shap_single = shap_explainer(
    X_single_shap
)

print("Single-Instance SHAP Verification")
print("=" * 60)

print(f"Input shape:       {X_single_shap.shape}")
print(f"SHAP values shape:  {shap_single.values.shape}")

print(f"\nInstance ID:       {shap_instance_ids[0]}")
print(f"True label:        {y_shap[0]:.0f}")

# Model prediction
single_prediction = shap_predict(X_single_shap)[0]

print(f"Model probability: {single_prediction:.6f}")

# Expected value / base value
print(f"Base value:        {shap_single.base_values[0]}")

# Feature attribution check
print("\nFeature attributions:")
for feature, value in zip(
    X_shap.columns,
    shap_single.values[0]
):
    print(f"{feature:<25} {value:+.8f}")

# ------------------------------------------------------------
# Additivity check
# ------------------------------------------------------------

base_value = np.asarray(shap_single.base_values[0]).item()
shap_sum = shap_single.values[0].sum()

print("\nAdditivity Check")
print("=" * 60)
print(f"Base value + SHAP sum: {base_value + shap_sum:.6f}")
print(f"Model probability:     {single_prediction:.6f}")
print(
    f"Absolute difference:   "
    f"{abs((base_value + shap_sum) - single_prediction):.10f}"
)

PermutationExplainer explainer: 2it [00:10, 10.39s/it]               

Single-Instance SHAP Verification
Input shape:       (1, 21)
SHAP values shape:  (1, 21)

Instance ID:       529
True label:        0
Model probability: 0.501891
Base value:        0.31813145988620817

Feature attributions:
HighBP                    +0.06000653
HighChol                  -0.04258002
CholCheck                 +0.00160859
BMI                       -0.10004776
Smoker                    +0.00501761
Stroke                    -0.00049158
HeartDiseaseorAttack      -0.00359675
PhysActivity              -0.00182458
Fruits                    +0.00323367
Veggies                   +0.00705422
HvyAlcoholConsump         +0.00311083
AnyHealthcare             +0.00100250
NoDocbcCost               +0.00056265
GenHlth                   +0.19585650
MentHlth                  +0.00523255
PhysHlth                  -0.02010542
DiffWalk                  -0.00564787
Sex                       +0.01365392
Age                       +0.03682720
Education                 +0.00081586
Income          

In [95]:
# ============================================================
# STEP 2D — SHAP on All 600 Sampled Instances
# ============================================================

print("Starting SHAP computation for 600 instances...")
print("=" * 60)

# Convert the 600 sampled instances to NumPy
X_shap_array = X_shap.to_numpy(dtype=np.float32)

# Compute SHAP explanations
shap_explanation = shap_explainer(
    X_shap_array
)

# Extract SHAP values
shap_values = shap_explanation.values

print("\nSHAP Computation Completed")
print("=" * 60)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected shape:    (600, 21)")

# ------------------------------------------------------------
# Verify SHAP output
# ------------------------------------------------------------

print("\nVerification")
print("=" * 60)

print(f"Number of instances: {shap_values.shape[0]}")
print(f"Number of features:  {shap_values.shape[1]}")

print(
    f"Contains NaN: "
    f"{np.isnan(shap_values).any()}"
)

print(
    f"Contains Inf: "
    f"{np.isinf(shap_values).any()}"
)

# ------------------------------------------------------------
# Save SHAP values
# ------------------------------------------------------------

np.save(
    "shap_values.npy",
    shap_values
)

# Save corresponding instance IDs
np.save(
    "shap_instance_ids.npy",
    shap_instance_ids
)

print("\nFiles Saved")
print("=" * 60)
print("shap_values.npy")
print("shap_instance_ids.npy")

Starting SHAP computation for 600 instances...


PermutationExplainer explainer: 601it [00:11,  7.22it/s]                         


SHAP Computation Completed
SHAP values shape: (600, 21)
Expected shape:    (600, 21)

Verification
Number of instances: 600
Number of features:  21
Contains NaN: False
Contains Inf: False

Files Saved
shap_values.npy
shap_instance_ids.npy


In [96]:
# ============================================================
# STEP 3A — Prepare and Verify LIME Sample
# ============================================================

# Load the same stratified sample used for SHAP
lime_sample_df = pd.read_csv("stratified_samples.csv")

# Extract the exact 600 test instance IDs
lime_instance_ids = lime_sample_df["test_instance_id"].to_numpy()

# Retrieve the corresponding 21 preprocessed features
X_lime = X_test_processed.iloc[lime_instance_ids].copy()

# Retrieve corresponding true labels
y_lime = y_test_processed[lime_instance_ids]

print("LIME Sample Verification")
print("=" * 60)

print(f"Sample shape:        {lime_sample_df.shape}")
print(f"X_lime shape:        {X_lime.shape}")
print(f"y_lime shape:        {y_lime.shape}")
print(f"Unique instance IDs: {len(np.unique(lime_instance_ids))}")

# Verify exact same instances as SHAP
same_instances = np.array_equal(
    shap_instance_ids,
    lime_instance_ids
)

print(f"\nSame instances as SHAP: {same_instances}")

# Stratum distribution
print("\nStratum counts:")
print(
    lime_sample_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

# Check feature count
print(f"\nFeature count: {X_lime.shape[1]}")

# Check for invalid values
print(f"Contains NaN: {X_lime.isna().any().any()}")
print(f"Contains Inf: {np.isinf(X_lime.to_numpy()).any()}")

print("\nFirst 5 instance IDs:")
print(lime_instance_ids[:5])

LIME Sample Verification
Sample shape:        (600, 3)
X_lime shape:        (600, 21)
y_lime shape:        (600,)
Unique instance IDs: 600

Same instances as SHAP: True

Stratum counts:
uncertainty_stratum
LOW       200
MEDIUM    200
HIGH      200
Name: count, dtype: int64

Feature count: 21
Contains NaN: False
Contains Inf: False

First 5 instance IDs:
[ 529  850  905 1029 1041]


In [99]:
# ============================================================
# STEP 3B — Final LIME Explainer Verification
# ============================================================

print("LIME Explainer Verification")
print("=" * 60)

print(f"Explainer type: {type(lime_explainer).__name__}")

print(f"Number of feature names: "
      f"{len(lime_explainer.feature_names)}")

print(f"Feature names match X_lime: "
      f"{lime_explainer.feature_names == X_lime.columns.tolist()}")

print(f"Mode: {lime_explainer.mode}")

# ------------------------------------------------------------
# Verify LIME prediction function
# ------------------------------------------------------------

test_input = X_lime.iloc[[0]].to_numpy(dtype=np.float32)

test_prediction = lime_predict_proba(test_input)

print("\nPrediction Function Check")
print("-" * 60)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_prediction.shape}")
print(f"Output:       {test_prediction}")

# ------------------------------------------------------------
# Probability checks
# ------------------------------------------------------------

probability_sum = test_prediction.sum(axis=1)

print(f"\nProbabilities sum to 1: "
      f"{np.allclose(probability_sum, 1.0)}")

print(f"Contains NaN: "
      f"{np.isnan(test_prediction).any()}")

print(f"Contains Inf: "
      f"{np.isinf(test_prediction).any()}")

# ------------------------------------------------------------
# Check probability values are valid
# ------------------------------------------------------------

print(
    f"All probabilities between 0 and 1: "
    f"{np.all((test_prediction >= 0) & (test_prediction <= 1))}"
)

LIME Explainer Verification
Explainer type: LimeTabularExplainer
Number of feature names: 21
Feature names match X_lime: True
Mode: classification

Prediction Function Check
------------------------------------------------------------
Input shape:  (1, 21)
Output shape: (1, 2)
Output:       [[0.4981094 0.5018906]]

Probabilities sum to 1: True
Contains NaN: False
Contains Inf: False
All probabilities between 0 and 1: True


In [100]:
# ============================================================
# STEP 3C — Single-Instance LIME Validation
# ============================================================

# Select the first sampled instance
lime_test_instance = X_lime.iloc[0].to_numpy(dtype=np.float32)

# Generate LIME explanation
lime_single_explanation = lime_explainer.explain_instance(
    lime_test_instance,
    lime_predict_proba,
    num_features=21
)

# Extract feature contributions
lime_single_contributions = lime_single_explanation.as_list()

print("Single-Instance LIME Validation")
print("=" * 60)

print(f"Instance ID:       {lime_instance_ids[0]}")
print(f"True label:        {y_lime[0]:.0f}")

# Model probability
lime_prediction = lime_predict_proba(
    lime_test_instance.reshape(1, -1)
)[0]

print(f"No Diabetes probability: {lime_prediction[0]:.6f}")
print(f"Diabetes probability:    {lime_prediction[1]:.6f}")

print(f"\nNumber of LIME features explained: "
      f"{len(lime_single_contributions)}")

print("\nFeature Contributions")
print("-" * 60)

for feature, contribution in lime_single_contributions:
    print(f"{feature:<30} {contribution:+.8f}")

# ------------------------------------------------------------
# Explanation object verification
# ------------------------------------------------------------

print("\nLIME Explanation Object")
print("-" * 60)
print(f"Explanation type: "
      f"{type(lime_single_explanation).__name__}")

print(
    f"Available labels: "
    f"{lime_single_explanation.available_labels()}"
)

Single-Instance LIME Validation
Instance ID:       529
True label:        0
No Diabetes probability: 0.498109
Diabetes probability:    0.501891

Number of LIME features explained: 21

Feature Contributions
------------------------------------------------------------
GenHlth                        +0.10078896
BMI                            +0.10066681
Age                            +0.07506873
HighBP                         +0.06135176
HighChol                       +0.04978108
CholCheck                      +0.03721505
HeartDiseaseorAttack           +0.02917695
Income                         -0.02780905
HvyAlcoholConsump              -0.02750253
MentHlth                       -0.02713350
Sex                            +0.01945961
DiffWalk                       +0.01363036
Stroke                         +0.01232809
Education                      -0.01231915
AnyHealthcare                  +0.00575146
PhysActivity                   -0.00387397
NoDocbcCost                    +0.00335387
Fr